# Processo de Limpeza e Organização de Dados - Justiça do Trabalho

Este notebook documenta as etapas de carregamento, exploração, limpeza e transformação da base de dados histórica da Justiça do Trabalho para viabilizar um dashboard em Streamlit.

### Objetivos:
1. Carregar os dados originais (`data/Base de Dados JT Historico - CSV.csv`) com a codificação e delimitadores corretos.
2. Filtrar apenas as colunas de interesse: `Variavel`, `Ano`, `Instancia`, `Regiao Judiciaria` e `Quantidade`.
3. Investigar e tratar valores nulos (especialmente para a instância 'TST', que não possui Região Judiciária delimitada).
4. Corrigir erros de digitação e inconsistências na coluna `Variavel`.
5. Limpar a formatação da coluna `Quantidade` (que possui formato de número em português com pontos e vírgulas) e convertê-la para números inteiros.
6. Salvar os dados limpos em formato Parquet para maior eficiência de leitura no dashboard.

In [ ]:
import pandas as pd
import os


## 1. Carregamento e Inspeção Inicial

O arquivo original possui codificação `latin1` (ou `cp1252`) e utiliza ponto e vírgula (`;`) como separador. Vamos carregá-lo e analisar o formato dos dados.

In [ ]:
csv_path = "data/Base de Dados JT Historico - CSV.csv"
df = pd.read_csv(csv_path, sep=";", encoding="latin1", low_memory=False)
print(f"Dimensões originais: {df.shape}")
print("Colunas disponíveis:", df.columns.tolist())
df.head()


## 2. Seleção de Colunas de Interesse

Conforme especificado, nos interessam apenas as colunas: `Variavel`, `Ano`, `Instancia`, `Regiao Judiciaria` e `Quantidade`.

In [ ]:
cols_of_interest = ["Variavel", "Ano", "Instancia", "Regiao Judiciaria", "Quantidade"]
df = df[cols_of_interest]
print(f"Dimensões após seleção de colunas: {df.shape}")
df.info()


## 3. Tratamento de Valores Nulos

Vamos analisar a presença de valores nulos em cada uma das colunas.

In [ ]:
print("Valores nulos por coluna antes do tratamento:")
print(df.isna().sum())

# Analisando a instância TST e a ausência de Região Judiciária
print("\nDistribuição de instâncias para Região Judiciária nula:")
print(df[df["Regiao Judiciaria"].isna()]["Instancia"].value_counts(dropna=False))

# A maioria esmagadora das Regiões Judiciárias nulas refere-se à instância 'TST' (que atua em âmbito nacional).
# Vamos preencher essa informação como 'TST (Nacional)'.
df.loc[df["Instancia"] == "TST", "Regiao Judiciaria"] = df.loc[
    df["Instancia"] == "TST", "Regiao Judiciaria"
].fillna("TST (Nacional)")

# Para as outras colunas essenciais (Variavel, Ano, Instancia, Quantidade) ou se sobrar alguma Região Judiciária nula (que seriam apenas 18 linhas da 1ª Instância),
# vamos descartar as linhas, pois são uma fração ínfima do dataset.
df = df.dropna(subset=["Variavel", "Ano", "Instancia", "Regiao Judiciaria", "Quantidade"])
print(f"\nDimensões após tratar nulos: {df.shape}")


## 4. Padronização e Correção das Variáveis

Podemos verificar que existem pequenas variações causadas por problemas de codificação e espaços na coluna `Variavel` (ex: `Arrecada  o`, `Res duo` e `Tempo M dio`). Vamos remover espaços extras nas pontas e padronizar os nomes das variáveis.

In [ ]:
df["Variavel"] = df["Variavel"].astype(str).str.strip()
df["Instancia"] = df["Instancia"].astype(str).str.strip()
df["Regiao Judiciaria"] = df["Regiao Judiciaria"].astype(str).str.strip()

var_mapping = {"Arrecada  o": "Arrecadação", "Res duo": "Resíduo", "Tempo M dio": "Tempo Médio"}
df["Variavel"] = df["Variavel"].replace(var_mapping)

print("Frequência das variáveis após padronização:")
print(df["Variavel"].value_counts())


## 5. Limpeza e Conversão de Tipos (`Quantidade` e `Ano`)

A coluna `Quantidade` está como texto devido ao formato brasileiro de números (ex: `20.322.288,00` ou `2.666,65`). Devemos remover os pontos de milhares, substituir a vírgula por ponto para decimais, converter em float, e por fim converter para inteiro (arredondando as frações para manter coerência com a quantidade de processos, conforme solicitado).
O `Ano` também será convertido de float para inteiro.

In [ ]:
# Remover formatação de milhar (.) e ajustar separador decimal (,)
qty_cleaned = (
    df["Quantidade"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)
df["Quantidade"] = pd.to_numeric(qty_cleaned, errors="coerce")

# Remover linhas que falharam na conversão (ex: vazias)
df = df.dropna(subset=["Quantidade"])

# Converter Quantidade para inteiro
df["Quantidade"] = df["Quantidade"].round().astype(int)

# Converter Ano para inteiro
df["Ano"] = df["Ano"].astype(int)

print("Tipos de dados finais:")
print(df.dtypes)
df.head()


## 6. Exportação para Formato Parquet

O formato Parquet é colunar e comprimido, ideal para o Streamlit ler os dados rapidamente sem precisar processar todo o CSV de 72MB a cada execução.

In [ ]:
output_parquet = "data/Base_de_Dados_JT_Historico_Limpa.parquet"
df.to_parquet(output_parquet, index=False)
print(f"Dados exportados com sucesso para: {output_parquet}")
csv_size = os.path.getsize(csv_path) / (1024 * 1024)
parquet_size = os.path.getsize(output_parquet) / (1024 * 1024)
print(f"Tamanho do CSV original: {csv_size:.2f} MB")
print(f"Tamanho do Parquet limpo: {parquet_size:.2f} MB")
print(f"Redução de tamanho: {(1 - parquet_size / csv_size) * 100:.2f}%")
